<a href="https://colab.research.google.com/github/giTan7/Adaptive-Stress-Monitoring-on-Wearable-Devices/blob/main/WESAD_CL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==== CONFIG ====
CSV_UNLABELED = "/content/all_subjects_unlabelled.csv"   # unlabeled data for pretraining
CSV_LABELLED  = "/content/generalized_labelled.csv"     # labeled data for general finetune
CSV_FINETUNE_SUBJECTWISE = "/content/finetune_labelled.csv"  # per-subject finetune data

ENCODER_SAVE_DIR    = "/content/encoder_package"
CLASSIFIER_SAVE_DIR = "/content/classifier_package"

# Windowing (15s @ 1Hz, from paper)
WINDOW_SEC  = 15
STRIDE_SEC  = 15
FS          = 1
TW          = WINDOW_SEC * FS
SH          = STRIDE_SEC * FS

# Training
BATCH_SIZE_PRETRAIN = 128
EPOCHS_PRETRAIN     = 15
LR_PRETRAIN         = 1e-3
TEMPERATURE         = 0.1
BATCH_SIZE_CLASSIF  = 64
EPOCHS_CLASSIF      = 30

SEED = 42

import os, json, random, gc
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import joblib
from tqdm import tqdm

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

os.makedirs(ENCODER_SAVE_DIR, exist_ok=True)
os.makedirs(CLASSIFIER_SAVE_DIR, exist_ok=True)


Using device: cuda


In [ ]:
# pick numeric signal columns (exclude labels/metadata)
def pick_numeric_columns(df):
    drop_like = {"stress_class","stressclass","task","sessionphase","activity","subject_id"}
    cols = []
    for c in df.columns:
        lc = c.lower()
        if any(k in lc for k in drop_like): continue
        if pd.api.types.is_numeric_dtype(df[c]): cols.append(c)
    return cols

def windows_from_array(arr, win=TW, stride=SH):
    starts = np.arange(0, len(arr)-win+1, stride)
    Xw = np.stack([arr[s:s+win].T for s in starts])  # [N, C, T]
    return Xw, starts

# augmentations (paper: scaling, noise, time-warp, masking, channel dropout)
def time_warp(x, max_warp=0.1):
    C,T = x.shape
    alpha = np.random.uniform(-max_warp,max_warp)
    scale = 1.0+alpha
    new_idx = np.linspace(0,T-1,int(np.round(T*scale)))
    warped = np.zeros_like(x)
    for c in range(C):
        vals = np.interp(new_idx,np.arange(T),x[c])
        warped[c] = np.interp(np.linspace(0,len(vals)-1,T),np.arange(len(vals)),vals)
    return warped

def augment_view(x):
    x = x.copy().astype(np.float32)
    C,T = x.shape
    x *= np.random.uniform(0.9,1.1)  # scaling
    x += np.random.normal(0,0.1,x.shape).astype(np.float32)  # noise
    if np.random.rand()<0.5: x = time_warp(x)                # time warp
    mask = np.random.rand(*x.shape)<0.15                     # feature mask
    x[mask] = 0
    if np.random.rand()<0.5:                                 # channel dropout
        drop_ch = np.random.choice(C,1)
        x[drop_ch]=0
    return x


In [ ]:
# unlabeled dataset for SimCLR
class SimCLRWindows(Dataset):
    def __init__(self, X):
        self.X = X.astype(np.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        v1,v2 = augment_view(self.X[i]), augment_view(self.X[i])
        return torch.from_numpy(v1), torch.from_numpy(v2)

# labeled dataset for embeddings + context + labels
class EmbCtxDataset(Dataset):
    def __init__(self,H,C,y):
        self.H=torch.tensor(H,dtype=torch.float32)
        self.C=torch.tensor(C,dtype=torch.float32)
        self.y=torch.tensor(y,dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self,i): return self.H[i],self.C[i],self.y[i]


In [ ]:
class EncoderCNNLSTM(nn.Module):
    def __init__(self,in_channels,proj_dim=128,lstm_hidden=64,dropout=0.2):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv1d(in_channels,32,kernel_size=5,padding=2),nn.ReLU(),
            nn.Conv1d(32,64,kernel_size=3,padding=1),nn.ReLU(),
            nn.Conv1d(64,64,kernel_size=3,padding=1),nn.ReLU()
        )
        self.lstm=nn.LSTM(64,lstm_hidden,batch_first=True,bidirectional=True,dropout=dropout)
        self.proj=nn.Sequential(
            nn.Linear(2*lstm_hidden,128),nn.ReLU(),nn.Linear(128,proj_dim)
        )
    def forward(self,x):
        h=self.net(x)             # [B,64,T]
        h=h.permute(0,2,1)        # [B,T,64]
        h,_=self.lstm(h)          # [B,T,128]
        h=h.mean(1)               # [B,128]
        z=self.proj(h)            # [B,proj_dim]
        return F.normalize(z,dim=1)


In [ ]:
def nt_xent_loss(z1,z2,temp=TEMPERATURE):
    N=z1.size(0)
    z=torch.cat([z1,z2],dim=0)        # [2N,D]
    sim=z@z.T/temp
    sim.fill_diagonal_(-9e15)
    targets=(torch.arange(2*N)+(N))%(2*N)
    return F.cross_entropy(sim,targets.to(z1.device))


In [ ]:
# load unlabeled
df_un=pd.read_csv(CSV_UNLABELED)
sig_cols=pick_numeric_columns(df_un)
scaler=StandardScaler()
X=scaler.fit_transform(df_un[sig_cols].astype(float))
X=pd.DataFrame(X).interpolate(limit_direction="both").fillna(0).values
Xw,_=windows_from_array(X,TW,SH)
print("Unlabeled windows:",Xw.shape)

# dataset/dataloader
ds=SimCLRWindows(Xw)
dl=DataLoader(ds,batch_size=BATCH_SIZE_PRETRAIN,shuffle=True,drop_last=True)

# encoder
encoder=EncoderCNNLSTM(in_channels=Xw.shape[1]).to(DEVICE)
opt=torch.optim.Adam(encoder.parameters(),lr=LR_PRETRAIN)

for ep in range(1,EPOCHS_PRETRAIN+1):
    losses=[]
    for v1,v2 in dl:
        v1,v2=v1.to(DEVICE),v2.to(DEVICE)
        z1,z2=encoder(v1),encoder(v2)
        loss=nt_xent_loss(z1,z2)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    print(f"Epoch {ep}: loss {np.mean(losses):.4f}")

# save encoder + scaler
torch.save(encoder.state_dict(),f"{ENCODER_SAVE_DIR}/encoder.pth")
joblib.dump(scaler,f"{ENCODER_SAVE_DIR}/scaler.pkl")
with open(f"{ENCODER_SAVE_DIR}/sig_cols.json","w") as f: json.dump(sig_cols,f)


Unlabeled windows: (5791, 13, 15)


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(


Epoch 1: loss 2.8343
Epoch 2: loss 1.1249
Epoch 3: loss 0.8447
Epoch 4: loss 0.7512
Epoch 5: loss 0.6998
Epoch 6: loss 0.6823
Epoch 7: loss 0.6418
Epoch 8: loss 0.6301
Epoch 9: loss 0.5855
Epoch 10: loss 0.6093
Epoch 11: loss 0.5672
Epoch 12: loss 0.5775
Epoch 13: loss 0.5611
Epoch 14: loss 0.5545
Epoch 15: loss 0.5412


In [ ]:
class GatedFusionClassifier(nn.Module):
    def __init__(self,emb_dim,ctx_dim,num_classes):
        super().__init__()
        self.ctx_proj=nn.Linear(ctx_dim,emb_dim)
        self.gate=nn.Linear(emb_dim*2,emb_dim)
        self.classifier=nn.Sequential(
            nn.Linear(emb_dim,128),nn.ReLU(),nn.Dropout(0.2),
            nn.Linear(128,num_classes)
        )
    def forward(self,h,c):
        cproj=self.ctx_proj(c)
        g=torch.sigmoid(self.gate(torch.cat([h,cproj],dim=1)))
        fused=g*h+(1-g)*cproj
        return self.classifier(fused)


In [ ]:
# ==== CELL 8: General Classifier Training & Saving ====

from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
import joblib

# Load labeled data
df_lab = pd.read_csv(CSV_LABELLED)
sig_cols = json.load(open(f"{ENCODER_SAVE_DIR}/sig_cols.json"))
scaler = joblib.load(f"{ENCODER_SAVE_DIR}/scaler.pkl")

# --- preprocess signals ---
X = scaler.transform(df_lab[sig_cols].astype(float))
X = pd.DataFrame(X).interpolate(limit_direction="both").fillna(0).values
Xw, starts = windows_from_array(X, TW, SH)

# --- majority vote label per window ---
y = df_lab["Stress_Class"].values
win_labels = [np.bincount(y[s:s+TW].astype(int)).argmax() for s in starts]

# --- preprocess context ---
context_cols = [c for c in ["Task", "SessionPhase", "Activity"] if c in df_lab.columns]
if context_cols:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)  # ensures dense matrix
    C = ohe.fit_transform(df_lab[context_cols].astype(str))
else:
    ohe = None
    C = np.zeros((len(df_lab), 1))

# force to dense float32
C = np.array(C, dtype=np.float32)

# align with windowing
Cw = np.stack([C[s] for s in starts]).astype(np.float32)

# --- compute encoder embeddings (frozen) ---
encoder.load_state_dict(torch.load(f"{ENCODER_SAVE_DIR}/encoder.pth", map_location=DEVICE))
encoder.to(DEVICE).eval()
with torch.no_grad():
    H = torch.cat([
        encoder(torch.tensor(Xw[i:i+64], dtype=torch.float32).to(DEVICE))
        for i in range(0, len(Xw), 64)
    ]).cpu().numpy()

# --- dataset/split ---
le = LabelEncoder()
y_enc = le.fit_transform(win_labels)
ds = EmbCtxDataset(H, Cw, y_enc)
train_idx, val_idx = train_test_split(np.arange(len(ds)), test_size=0.2,
                                      stratify=y_enc, random_state=SEED)
train_dl = DataLoader(torch.utils.data.Subset(ds, train_idx), batch_size=BATCH_SIZE_CLASSIF, shuffle=True)
val_dl = DataLoader(torch.utils.data.Subset(ds, val_idx), batch_size=BATCH_SIZE_CLASSIF)

# --- init classifier ---
model = GatedFusionClassifier(H.shape[1], Cw.shape[1], len(le.classes_)).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()

# --- training ---
for ep in range(1, EPOCHS_CLASSIF+1):
    model.train(); tloss=[]; tacc=[]
    for Ht, Ct, yt in train_dl:
        Ht, Ct, yt = Ht.to(DEVICE), Ct.to(DEVICE), yt.to(DEVICE)
        out = model(Ht, Ct)
        loss = crit(out, yt)
        opt.zero_grad(); loss.backward(); opt.step()
        tloss.append(loss.item())
        tacc.append((out.argmax(1) == yt).float().mean().item())
    model.eval(); valacc=[]
    with torch.no_grad():
        for Hv, Cv, yv in val_dl:
            Hv, Cv, yv = Hv.to(DEVICE), Cv.to(DEVICE), yv.to(DEVICE)
            o = model(Hv, Cv)
            valacc.append((o.argmax(1) == yv).float().mean().item())
    print(f"Epoch {ep}: train_acc {np.mean(tacc):.3f} val_acc {np.mean(valacc):.3f}")

# --- save everything ---
os.makedirs(CLASSIFIER_SAVE_DIR, exist_ok=True)
torch.save(model.state_dict(), f"{CLASSIFIER_SAVE_DIR}/classifier_general.pth")
joblib.dump(le, f"{CLASSIFIER_SAVE_DIR}/label_encoder.pkl")
if ohe:
    joblib.dump(ohe, f"{CLASSIFIER_SAVE_DIR}/context_ohe.pkl")

print(f"\nGeneral classifier saved to {CLASSIFIER_SAVE_DIR}")


Epoch 1: train_acc 0.426 val_acc 0.440
Epoch 2: train_acc 0.402 val_acc 0.440
Epoch 3: train_acc 0.473 val_acc 0.400
Epoch 4: train_acc 0.566 val_acc 0.440
Epoch 5: train_acc 0.512 val_acc 0.420
Epoch 6: train_acc 0.480 val_acc 0.480
Epoch 7: train_acc 0.605 val_acc 0.440
Epoch 8: train_acc 0.539 val_acc 0.440
Epoch 9: train_acc 0.629 val_acc 0.440
Epoch 10: train_acc 0.590 val_acc 0.480
Epoch 11: train_acc 0.539 val_acc 0.460
Epoch 12: train_acc 0.500 val_acc 0.460
Epoch 13: train_acc 0.551 val_acc 0.460
Epoch 14: train_acc 0.598 val_acc 0.500
Epoch 15: train_acc 0.629 val_acc 0.500
Epoch 16: train_acc 0.539 val_acc 0.500
Epoch 17: train_acc 0.652 val_acc 0.520
Epoch 18: train_acc 0.633 val_acc 0.500
Epoch 19: train_acc 0.605 val_acc 0.540
Epoch 20: train_acc 0.582 val_acc 0.500
Epoch 21: train_acc 0.645 val_acc 0.560
Epoch 22: train_acc 0.586 val_acc 0.540
Epoch 23: train_acc 0.703 val_acc 0.540
Epoch 24: train_acc 0.680 val_acc 0.560
Epoch 25: train_acc 0.715 val_acc 0.540
Epoch 26:

In [ ]:
# ==== CELL 9: Subject-wise Fine-tuning (robust) ====

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# Load finetune dataset
df_sub = pd.read_csv(CSV_FINETUNE_SUBJECTWISE)
sig_cols = json.load(open(f"{ENCODER_SAVE_DIR}/sig_cols.json"))
scaler = joblib.load(f"{ENCODER_SAVE_DIR}/scaler.pkl")
le = joblib.load(f"{CLASSIFIER_SAVE_DIR}/label_encoder.pkl")
ohe = joblib.load(f"{CLASSIFIER_SAVE_DIR}/context_ohe.pkl") if os.path.exists(f"{CLASSIFIER_SAVE_DIR}/context_ohe.pkl") else None

results = []

# loop through subjects
for sid, df_s in df_sub.groupby("Subject_ID"):
    print(f"\n=== Fine-tuning for subject {sid} ===")

    # --- signals ---
    X = scaler.transform(df_s[sig_cols].astype(float))
    X = pd.DataFrame(X).interpolate(limit_direction="both").fillna(0).values
    Xw, starts = windows_from_array(X, TW, SH)

    # --- labels ---
    y = df_s["Stress_Class"].values
    win_labels = [np.bincount(y[s:s+TW].astype(int)).argmax() for s in starts]
    y_enc = le.transform(win_labels)  # same label encoder as general

    # --- context ---
    if ohe:
        context_cols = ohe.feature_names_in_
        C = ohe.transform(df_s[context_cols].astype(str))
    else:
        C = np.zeros((len(df_s), 1))
    Cw = np.stack([C[s] for s in starts]).astype(np.float32)

    # --- encoder embeddings (frozen) ---
    encoder.load_state_dict(torch.load(f"{ENCODER_SAVE_DIR}/encoder.pth", map_location=DEVICE))
    encoder.eval()
    with torch.no_grad():
        H = torch.cat([encoder(torch.tensor(Xw[i:i+64], dtype=torch.float32).to(DEVICE))
                       for i in range(0, len(Xw), 64)]).cpu().numpy()

    # --- dataset/split ---
    ds = EmbCtxDataset(H, Cw, y_enc)
    try:
        # stratified split if possible
        train_idx, val_idx = train_test_split(np.arange(len(ds)), test_size=0.2,
                                              stratify=y_enc, random_state=SEED)
    except ValueError:
        # fallback: random split
        train_idx, val_idx = train_test_split(np.arange(len(ds)), test_size=0.2,
                                              random_state=SEED)
        print(f"  Warning: stratify skipped for subject {sid} (not enough classes)")

    train_dl = DataLoader(torch.utils.data.Subset(ds, train_idx), batch_size=BATCH_SIZE_CLASSIF, shuffle=True)
    val_dl   = DataLoader(torch.utils.data.Subset(ds, val_idx), batch_size=BATCH_SIZE_CLASSIF)

    # --- init classifier from general weights ---
    model = GatedFusionClassifier(H.shape[1], Cw.shape[1], len(le.classes_)).to(DEVICE)
    model.load_state_dict(torch.load(f"{CLASSIFIER_SAVE_DIR}/classifier_general.pth", map_location=DEVICE))
    opt = torch.optim.Adam(model.parameters(), lr=1e-4)  # smaller LR for fine-tune
    crit = nn.CrossEntropyLoss()

    # --- train ---
    best_val = 0
    for ep in range(1, 11):  # small epochs per subject
        model.train(); tloss=[]; tacc=[]
        for Ht,Ct,yt in train_dl:
            Ht,Ct,yt = Ht.to(DEVICE),Ct.to(DEVICE),yt.to(DEVICE)
            out = model(Ht,Ct)
            loss = crit(out,yt)
            opt.zero_grad(); loss.backward(); opt.step()
            tloss.append(loss.item())
            tacc.append((out.argmax(1)==yt).float().mean().item())
        # val
        model.eval(); y_true=[]; y_pred=[]
        with torch.no_grad():
            for Hv,Cv,yv in val_dl:
                Hv,Cv,yv = Hv.to(DEVICE),Cv.to(DEVICE),yv.to(DEVICE)
                o = model(Hv,Cv)
                y_true.extend(yv.cpu().numpy())
                y_pred.extend(o.argmax(1).cpu().numpy())
        val_acc = accuracy_score(y_true,y_pred)
        val_f1  = f1_score(y_true,y_pred,average="macro")
        print(f"Epoch {ep}: train_acc {np.mean(tacc):.3f} val_acc {val_acc:.3f} val_f1 {val_f1:.3f}")
        if val_acc > best_val:
            best_val = val_acc
            torch.save(model.state_dict(), f"{CLASSIFIER_SAVE_DIR}/classifier_subject_{sid}.pth")

    # log results
    results.append({
        "subject": sid,
        "val_acc": val_acc,
        "val_f1": val_f1,
        "precision": precision_score(y_true,y_pred,average="macro"),
        "recall": recall_score(y_true,y_pred,average="macro")
    })

# save metrics summary
pd.DataFrame(results).to_csv(f"{CLASSIFIER_SAVE_DIR}/subjectwise_results.csv", index=False)
print("\nSubject-wise fine-tuning complete. Results saved.")


In [2]:
!git clone https://github.com/giTan7/Adaptive-Stress-Monitoring-on-Wearable-Devices.git

Cloning into 'Adaptive-Stress-Monitoring-on-Wearable-Devices'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.


In [4]:
!ls -a


.  ..  Adaptive-Stress-Monitoring-on-Wearable-Devices  .config	sample_data


In [9]:
!pwd
!ls

/content/Adaptive-Stress-Monitoring-on-Wearable-Devices
README


In [6]:
%cd Adaptive-Stress-Monitoring-on-Wearable-Devices

/content/Adaptive-Stress-Monitoring-on-Wearable-Devices


In [10]:
!git config --global user.name "giTan7"
!git config --global user.email "tarannum.ara17@gmail.com"


!git add .
!git commit -m "initial commit"
!git push origin main

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
fatal: could not read Username for 'https://github.com': No such device or address
